In [ ]:
import random

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
ARROWS = {0: '←', 1: '↓', 2: '→', 3: '↑'}

CONFIG = {
    'seed': 0,
    'num_episodes': 3000,
    'alpha': 0.1,
    'gamma': 0.99,
    'epsilon': 1.0,
    'epsilon_decay': 0.999,
    'epsilon_min': 0.1,
    'max_steps': 100,
}


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)


def make_env(is_slippery, seed=None):
    env = gym.make('FrozenLake-v1', is_slippery=is_slippery)
    if seed is not None:
        env.reset(seed=seed)
    return env


def moving_average(x, w=100):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x.copy()
    kernel = np.ones(w, dtype=float) / w
    return np.convolve(x, kernel, mode='valid')

In [ ]:
env = make_env(is_slippery=False, seed=CONFIG['seed'])
state, info = env.reset()
print('초기 상태:', state)
print('상태 개수:', env.observation_space.n)
print('행동 개수:', env.action_space.n)
print('맵:')
for row in env.unwrapped.desc.astype(str):
    print(' '.join(row))
env.close()

In [ ]:
def evaluate_greedy(Q, is_slippery, seed=0, n_eval_episodes=100, max_steps=100):
    env = make_env(is_slippery=is_slippery, seed=seed)
    success = 0

    for episode in range(n_eval_episodes):
        state, _ = env.reset(seed=seed + 1000 + episode)

        for _ in range(max_steps):
            action = int(np.argmax(Q[state]))
            next_state, reward, terminated, truncated, _ = env.step(action)
            state = next_state

            if terminated or truncated:
                success += int(reward > 0)
                break

    env.close()
    return success / n_eval_episodes


def print_policy(Q, is_slippery=False, seed=0):
    env = make_env(is_slippery=is_slippery, seed=seed)
    desc = env.unwrapped.desc.astype(str)
    nrow, ncol = desc.shape
    rows = []

    for r in range(nrow):
        row = []
        for c in range(ncol):
            tile = desc[r, c]
            s = r * ncol + c
            if tile in ('S', 'H', 'G'):
                row.append(tile)
            else:
                row.append(ARROWS[int(np.argmax(Q[s]))])
        rows.append(' '.join(row))

    env.close()
    print('\n'.join(rows))

## 1. 구현

핵심은 target만 다르게 보는 것이다.
- SARSA: 실제 다음 행동 `a'`를 써서 `reward + gamma * Q(s', a')`
- Q-learning: 가장 큰 다음 Q값을 써서 `reward + gamma * max_a Q(s', a)`

In [ ]:
def epsilon_greedy(Q, state, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return int(np.argmax(Q[state]))


def train_sarsa(
    env,
    num_episodes=3000,
    alpha=0.1,
    gamma=0.99,
    epsilon=1.0,
    epsilon_decay=0.999,
    epsilon_min=0.1,
    max_steps=100,
):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    rewards = []
    eps = epsilon

    for episode in range(num_episodes):
        state, _ = env.reset()
        action = epsilon_greedy(Q, state, eps, n_actions)
        total_reward = 0.0

        for _ in range(max_steps):
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            if done:
                td_target = reward
                Q[state, action] += alpha * (td_target - Q[state, action])
                break

            next_action = epsilon_greedy(Q, next_state, eps, n_actions)
            td_target = reward + gamma * Q[next_state, next_action]
            Q[state, action] += alpha * (td_target - Q[state, action])

            state, action = next_state, next_action

        rewards.append(total_reward)
        eps = max(epsilon_min, eps * epsilon_decay)

    return Q, np.asarray(rewards, dtype=np.float32)


def train_q_learning(
    env,
    num_episodes=3000,
    alpha=0.1,
    gamma=0.99,
    epsilon=1.0,
    epsilon_decay=0.999,
    epsilon_min=0.1,
    max_steps=100,
):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    rewards = []
    eps = epsilon

    for episode in range(num_episodes):
        state, _ = env.reset()
        total_reward = 0.0

        for _ in range(max_steps):
            action = epsilon_greedy(Q, state, eps, n_actions)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            best_next = np.max(Q[next_state])
            td_target = reward if done else (reward + gamma * best_next)
            Q[state, action] += alpha * (td_target - Q[state, action])

            state = next_state
            if done:
                break

        rewards.append(total_reward)
        eps = max(epsilon_min, eps * epsilon_decay)

    return Q, np.asarray(rewards, dtype=np.float32)

## 2. 실행

이 셀은 총 4개 실험을 자동으로 돌린다.
- 학습 중 지표: 최근 100 episode 평균 보상
- 평가 지표: 학습 후 epsilon=0 greedy 100 episode 성공률

FrozenLake 4x4는 variance가 있어서 seed와 epsilon 설정에 따라 차이가 달라질 수 있다.

In [ ]:
experiments = [
    ('4x4 non-slippery', False, 'SARSA', train_sarsa),
    ('4x4 non-slippery', False, 'Q-learning', train_q_learning),
    ('4x4 slippery', True, 'SARSA', train_sarsa),
    ('4x4 slippery', True, 'Q-learning', train_q_learning),
]

results = []
histories = {}
policies = {}

for env_name, is_slippery, algo_name, train_fn in experiments:
    set_seed(CONFIG['seed'])
    env = make_env(is_slippery=is_slippery, seed=CONFIG['seed'])
    Q, rewards = train_fn(
        env,
        num_episodes=CONFIG['num_episodes'],
        alpha=CONFIG['alpha'],
        gamma=CONFIG['gamma'],
        epsilon=CONFIG['epsilon'],
        epsilon_decay=CONFIG['epsilon_decay'],
        epsilon_min=CONFIG['epsilon_min'],
        max_steps=CONFIG['max_steps'],
    )
    env.close()

    train_last100 = float(rewards[-100:].mean())
    greedy_success = evaluate_greedy(
        Q,
        is_slippery=is_slippery,
        seed=CONFIG['seed'],
        n_eval_episodes=100,
        max_steps=CONFIG['max_steps'],
    )

    results.append(
        {
            'environment': env_name,
            'algorithm': algo_name,
            'train avg reward(last100)': round(train_last100, 3),
            'greedy success rate(100eps)': round(float(greedy_success), 3),
        }
    )
    histories[(env_name, algo_name)] = rewards
    policies[(env_name, algo_name)] = Q

summary_df = pd.DataFrame(results)
summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, env_name in zip(axes, ['4x4 non-slippery', '4x4 slippery']):
    for algo_name in ['SARSA', 'Q-learning']:
        rewards = histories[(env_name, algo_name)]
        ax.plot(moving_average(rewards, w=100), label=algo_name)
    ax.set_title(env_name)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Average reward (window=100)')
    ax.legend()

plt.tight_layout()
plt.show()

for env_name in ['4x4 non-slippery', '4x4 slippery']:
    is_slippery = env_name == '4x4 slippery'
    for algo_name in ['SARSA', 'Q-learning']:
        print(f'=== {env_name} / {algo_name} ===')
        print_policy(policies[(env_name, algo_name)], is_slippery=is_slippery, seed=CONFIG['seed'])
        print()

summary_df

## 3. 짧은 글 과제 예시 답안

1. `G_t = r_(t+1) + gamma * G_(t+1)` 이 성립하는 이유:
   return은 지금 받은 즉시 보상과 그 다음 시점부터의 할인된 미래 return을 합친 값으로 정의되기 때문이다.

2. SARSA target이 `reward + gamma * Q(s', a')` 인 이유:
   SARSA는 실제로 현재 정책이 다음 상태에서 고를 행동 `a'`의 가치를 이어붙여 한 step TD target을 만든다.

3. Q-learning target이 `reward + gamma * max_a Q(s', a)` 인 이유:
   Q-learning은 다음 상태에서 가장 좋은 행동을 했다고 가정한 최적 action-value를 근사하려고 하기 때문이다.

4. SARSA가 on-policy인 이유:
   데이터를 모으는 정책과 update target에 쓰는 정책이 같은 epsilon-greedy 정책이기 때문이다.

5. Q-learning이 off-policy인 이유:
   실제 행동은 epsilon-greedy로 해도 update target은 별도의 greedy 정책 `max_a Q(s', a)`를 쓰기 때문이다.

해석 예시
- non-slippery에서는 둘 다 비교적 쉽게 좋은 정책을 찾는 경우가 많다.
- slippery에서는 variance가 커져서 seed에 따라 우열이 달라질 수 있지만, train 지표와 greedy eval 지표가 서로 다르게 보일 수 있다는 점이 중요하다.
- 이 분리가 DQN에서도 그대로 중요하다.